# Open Quantum Systems: Lindblad Master Equation for Coupled Harmonic Oscillators

### Overview & Pedagogical Design
In this notebook, **no open-quantum-system functions are pre-solved for you**. You will implement every step of a **[Gorini–Kossakowski–Sudarshan–Lindblad (GKSL) Master Equation](https://en.wikipedia.org/wiki/Lindbladian)** simulator for two coupled quantum harmonic oscillators across **12 granular `# TODO` functions**:

1. **Part 1 — Truncated Fock Basis & Composite Tensor-Product Operators (TODOs 1–2)**:
   - `build_single_mode_ladder_operators`: Construct $N \times N$ matrices for $\hat{a}, \hat{a}^\dagger, \hat{n}, \hat{x}, \hat{p}$ in the truncated Fock basis $\{|0\rangle, \dots, |N-1\rangle\}$.
   - `build_composite_bipartite_operators`: Lift single-mode operators to the composite Hilbert space $\mathcal{H}_1 \otimes \mathcal{H}_2$ (dimension $D = N_1 N_2$) via Kronecker products (`np.kron`).
2. **Part 2 — Coupled Harmonic Oscillator Hamiltonian $\hat{H}$ (TODO 3)**:
   - `build_coupled_ho_hamiltonian`: Construct the bipartite Hamiltonian under both the Rotating-Wave Approximation ($\hbar g(\hat{a}_1^\dagger\hat{a}_2 + \hat{a}_1\hat{a}_2^\dagger)$) and full coordinate coupling ($\hbar g(\hat{a}_1 + \hat{a}_1^\dagger)(\hat{a}_2 + \hat{a}_2^\dagger)$).
3. **Part 3 — Collapse Operator Matrices $\hat{L}_k$ & Constituent Parts of the Master Equation (TODOs 4–8)**:
   - `build_collapse_operators`: Construct the $D \times D$ collapse matrices for energy emission ($\sqrt{\kappa_j(n_{\text{th},j}+1)}\,\hat{a}_j$), thermal absorption ($\sqrt{\kappa_j n_{\text{th},j}}\,\hat{a}_j^\dagger$), and pure dephasing ($\sqrt{\gamma_{\phi,j}}\,\hat{n}_j$) ([QuTiP `mesolve` Guide](https://qutip.org/QuantumToolbox.jl/stable/users_guide/time_evolution/mesolve)).
   - `compute_unitary_commutator_part`: Evaluate $\mathcal{L}_{\text{unitary}}[\hat{\rho}] = -i[\hat{H}, \hat{\rho}]$.
   - `compute_quantum_jump_part`: Evaluate the quantum jump (recycling) sum $\mathcal{J}[\hat{\rho}] = \sum_k \hat{L}_k\hat{\rho}\hat{L}_k^\dagger$.
   - `compute_anticommutator_damping_part`: Evaluate the non-Hermitian damping sum $\mathcal{A}[\hat{\rho}] = -\frac{1}{2}\sum_k \{\hat{L}_k^\dagger\hat{L}_k, \hat{\rho}\}$.
   - `build_liouvillian_superoperator`: Construct the $D^2 \times D^2$ vectorized Liouvillian superoperator matrix $\mathcal{L}$ ([AIP Advances 9, 095019](https://pubs.aip.org/aip/adv/article/9/9/095019/1076014/Density-matrix-superoperator-for-periodic-quantum)).
4. **Part 4 — Density Matrix Propagation, Partial Trace & Steady-State Nullspace (TODOs 9–12)**:
   - `rk4_step_lindblad`: Advance $\hat{\rho}(t) \to \hat{\rho}(t + \Delta t)$ using 4th-order Runge–Kutta ([arXiv:2506.19238](https://arxiv.org/pdf/2506.19238)).
   - `partial_trace_and_entropy`: Reshape $\hat{\rho}$ into a 4-index tensor $\rho_{i_1, i_2;\, j_1, j_2}$, trace out Oscillator 2 ($\hat{\rho}_1 = \text{Tr}_2[\hat{\rho}]$), and evaluate subsystem von Neumann entropy $S(\hat{\rho}_1) = -\text{Tr}[\hat{\rho}_1 \ln \hat{\rho}_1]$.
   - `propagate_lindblad_trajectory`: Evolve $\hat{\rho}(t)$ over $t \in [0, t_{\max}]$ while recording $\langle\hat{n}_1(t)\rangle, \langle\hat{n}_2(t)\rangle$, purity $\text{Tr}[\hat{\rho}(t)^2]$, $S(\hat{\rho}_1(t))$, trace error, and minimum eigenvalue.
   - `compute_liouvillian_steady_state`: Extract the stationary density matrix $\hat{\rho}_{\text{ss}}$ from the zero-eigenvalue nullspace $\mathcal{L}|\hat{\rho}_{\text{ss}}\rangle\!\rangle = 0$ ([Phys. Rev. A 101, 042108](https://link.aps.org/doi/10.1103/PhysRevA.101.042108)).
5. **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_lindblad_checkpoints`)**.

In [ ]:
import numpy as np
from scipy.linalg import eigh, expm, null_space
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("Environment initialized. Implement TODOs 1-12 and run `run_lindblad_checkpoints()` in Part 5!")

## Part 1: Truncated Fock Basis & Composite Ladder Operators (TODOs 1–2)

$$\hat{a} = \sum_{n=1}^{N-1} \sqrt{n}\,|n-1\rangle\langle n|, \quad \hat{a}^\dagger = \hat{a}^\mathsf{T}, \quad \hat{n} = \hat{a}^\dagger\hat{a}, \quad \hat{x} = \frac{\hat{a}+\hat{a}^\dagger}{\sqrt{2}}, \quad \hat{p} = \frac{-i(\hat{a}-\hat{a}^\dagger)}{\sqrt{2}}$$
$$\hat{a}_1 = \hat{a}^{(1)} \otimes \hat{I}_{N_2}, \qquad \hat{a}_2 = \hat{I}_{N_1} \otimes \hat{a}^{(2)}$$

In [ ]:
def build_single_mode_ladder_operators(n_cutoff: int) -> dict[str, np.ndarray]:
    '''
    TODO 1: Construct single-mode harmonic oscillator matrices in a truncated Fock basis |0>..|n_cutoff-1>.
    Return a dict with keys 'a', 'a_dag', 'n', 'x', 'p', 'I' (all of dtype complex128).
    '''
    # TODO 1: Build `a` using np.diag(np.sqrt(np.arange(1, n_cutoff)), k=1), then form a_dag, n, x, p, and I.
    raise NotImplementedError("Complete TODO 1: build_single_mode_ladder_operators")


def build_composite_bipartite_operators(n1: int, n2: int) -> dict[str, np.ndarray | int]:
    '''
    TODO 2: Lift single-mode operators into H_1 (x) H_2 of dimension D = n1 * n2 using np.kron.
    Return a dict with keys 'a1', 'a1_dag', 'n1', 'x1', 'p1', 'a2', 'a2_dag', 'n2', 'x2', 'p2',
    'I', 'n1_dim', 'n2_dim', 'total_dim'.
    '''
    # TODO 2: Call build_single_mode_ladder_operators(n1) and (n2), and apply np.kron(..., I2) and np.kron(I1, ...).
    raise NotImplementedError("Complete TODO 2: build_composite_bipartite_operators")

## Part 2: Coupled Harmonic Oscillator Hamiltonian $\hat{H}$ (TODO 3)

$$\hat{H}_{\text{RWA}} = \omega_1 \hat{n}_1 + \omega_2 \hat{n}_2 + g\left(\hat{a}_1^\dagger \hat{a}_2 + \hat{a}_1 \hat{a}_2^\dagger\right), \qquad \hat{H}_{\text{full}} = \omega_1 \hat{n}_1 + \omega_2 \hat{n}_2 + g\left(\hat{a}_1 + \hat{a}_1^\dagger\right)\left(\hat{a}_2 + \hat{a}_2^\dagger\right)$$

In [ ]:
def build_coupled_ho_hamiltonian(
    ops: dict,
    omega1: float = 1.0,
    omega2: float = 1.0,
    g_coupling: float = 0.15,
    include_counter_rotating: bool = False,
) -> np.ndarray:
    '''
    TODO 3: Construct the D x D coupled harmonic oscillator Hamiltonian matrix.
    '''
    # TODO 3: Build H_0 = omega1 * n1 + omega2 * n2 and add either RWA hopping or full coordinate coupling.
    raise NotImplementedError("Complete TODO 3: build_coupled_ho_hamiltonian")

## Part 3: Collapse Operators $\hat{L}_k$, Master Equation Parts & Liouvillian Superoperator (TODOs 4–8)

$$\frac{d\hat{\rho}}{dt} = \underbrace{-i[\hat{H}, \hat{\rho}]}_{\text{Unitary (TODO 5)}} + \underbrace{\sum_k \hat{L}_k \hat{\rho} \hat{L}_k^\dagger}_{\text{Jump (TODO 6)}} + \underbrace{\left(-\frac{1}{2}\sum_k \left\{\hat{L}_k^\dagger \hat{L}_k, \hat{\rho}\right\}\right)}_{\text{Anti-commutator (TODO 7)}}$$

$$\mathcal{L} = -i\left(\hat{H} \otimes \hat{I} - \hat{I} \otimes \hat{H}^{\mathsf{T}}\right) + \sum_k \left[ \hat{L}_k \otimes \hat{L}_k^* - \frac{1}{2}\left(\hat{L}_k^\dagger \hat{L}_k \otimes \hat{I}\right) - \frac{1}{2}\left(\hat{I} \otimes (\hat{L}_k^\dagger \hat{L}_k)^{\mathsf{T}}\right) \right]$$

In [ ]:
def build_collapse_operators(
    ops: dict,
    kappa1: float = 0.06,
    kappa2: float = 0.03,
    n_th1: float = 0.15,
    n_th2: float = 0.02,
    gamma_phi1: float = 0.02,
    gamma_phi2: float = 0.01,
) -> list[np.ndarray]:
    '''
    TODO 4: Return the list of D x D collapse operator matrices [L_1, L_2, ...] for all active channels
    (emission sqrt(kappa*(n_th+1))*a, thermal absorption sqrt(kappa*n_th)*a_dag, dephasing sqrt(gamma_phi)*n).
    '''
    # TODO 4: Append each non-zero collapse operator matrix to a list and return it.
    raise NotImplementedError("Complete TODO 4: build_collapse_operators")


def compute_unitary_commutator_part(rho: np.ndarray, H: np.ndarray) -> np.ndarray:
    '''
    TODO 5: Compute the unitary von Neumann part -i [H, rho] = -i (H rho - rho H).
    '''
    # TODO 5: Return -1j * (H @ rho - rho @ H).
    raise NotImplementedError("Complete TODO 5: compute_unitary_commutator_part")


def compute_quantum_jump_part(rho: np.ndarray, c_ops: list[np.ndarray]) -> np.ndarray:
    '''
    TODO 6: Compute the quantum jump (recycling) sum J[rho] = sum_k L_k rho L_k^dagger.
    '''
    # TODO 6: Accumulate L_k @ rho @ L_k.conj().T over all collapse operators in c_ops.
    raise NotImplementedError("Complete TODO 6: compute_quantum_jump_part")


def compute_anticommutator_damping_part(rho: np.ndarray, c_ops: list[np.ndarray]) -> np.ndarray:
    '''
    TODO 7: Compute the non-Hermitian anti-commutator sum A[rho] = -0.5 * sum_k {L_k^dagger L_k, rho}.
    '''
    # TODO 7: Accumulate -0.5 * (L_k^dagger L_k @ rho + rho @ L_k^dagger L_k) over all L_k in c_ops.
    raise NotImplementedError("Complete TODO 7: compute_anticommutator_damping_part")


def build_liouvillian_superoperator(H: np.ndarray, c_ops: list[np.ndarray]) -> np.ndarray:
    '''
    TODO 8: Construct the D^2 x D^2 Liouvillian superoperator matrix L in row-vectorized convention
    such that (L @ rho.ravel()).reshape(D, D) equals the total Lindblad derivative d(rho)/dt.
    '''
    # TODO 8: Use np.kron to form the unitary superoperator and add each dissipator superoperator.
    raise NotImplementedError("Complete TODO 8: build_liouvillian_superoperator")

## Part 4: Density Matrix Propagation, Partial Trace & Steady-State Nullspace (TODOs 9–12)

In [ ]:
def rk4_step_lindblad(
    rho: np.ndarray, H: np.ndarray, c_ops: list[np.ndarray], dt: float
) -> np.ndarray:
    '''
    TODO 9: Advance density matrix `rho` by one time step `dt` using 4th-order Runge-Kutta
    and your functions from TODOs 5, 6, and 7.
    '''
    # TODO 9: Define drho_dt(r) = unitary(r) + jump(r) + anticomm(r), evaluate k1..k4, and return updated rho.
    raise NotImplementedError("Complete TODO 9: rk4_step_lindblad")


def partial_trace_and_entropy(
    rho: np.ndarray, n1: int, n2: int, eps: float = 1e-15
) -> tuple[np.ndarray, float]:
    '''
    TODO 10: Compute the reduced density matrix rho_1 = Tr_2[rho] of Oscillator 1
    by reshaping rho to (n1, n2, n1, n2) and contracting over Oscillator 2, then
    evaluate its von Neumann entropy S(rho_1) = -Tr[rho_1 ln(rho_1)]. Return (rho_1, S_vn).
    '''
    # TODO 10: Reshape rho to (n1, n2, n1, n2), contract via einsum("ikjk->ij", ...),
    #          and compute -sum(evals * log(evals)) from eigvalsh(rho_1).
    raise NotImplementedError("Complete TODO 10: partial_trace_and_entropy")


def propagate_lindblad_trajectory(
    rho_init: np.ndarray,
    H: np.ndarray,
    c_ops: list[np.ndarray],
    ops: dict,
    t_max: float = 60.0,
    n_steps: int = 500,
) -> dict[str, np.ndarray]:
    '''
    TODO 11: Propagate rho(t) over n_steps steps up to t_max using `rk4_step_lindblad`.
    Return a dict with keys 'times', 'n1', 'n2', 'purity', 'entropy_mode1', 'trace_error',
    'min_eigenvalue', and 'final_rho'.
    '''
    # TODO 11: Loop over steps, recording observables Tr(n1 @ rho), Tr(n2 @ rho), Tr(rho @ rho),
    #          S(rho_1), |Tr(rho) - 1|, and min(eigvalsh(rho)), and step rho via rk4_step_lindblad.
    raise NotImplementedError("Complete TODO 11: propagate_lindblad_trajectory")


def compute_liouvillian_steady_state(L_super: np.ndarray, dim: int) -> np.ndarray:
    '''
    TODO 12: Extract the stationary density matrix rho_ss from the zero-eigenvalue nullspace
    of L_super (using scipy.linalg.null_space), Hermitianize it, and normalize its trace to 1.
    '''
    # TODO 12: Find the null vector of L_super, reshape to (dim, dim), Hermitianize, and divide by trace.
    raise NotImplementedError("Complete TODO 12: compute_liouvillian_steady_state")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_lindblad_checkpoints()` below as you implement each `# TODO` function to test your code against exact analytical and matrix-exponential benchmarks.

In [2]:
def run_lindblad_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 12 #TODO functions in the Lindblad notebook.'''
    passed = 0
    total_todos = 12

    try:
        s_ops = build_single_mode_ladder_operators(5)
        assert s_ops["a"].shape == (5, 5) and np.allclose(np.diag(s_ops["n"]).real, np.arange(5))
        passed += 1
        print("[PASS] TODO 1 : build_single_mode_ladder_operators")

        ops = build_composite_bipartite_operators(5, 5)
        assert ops["a1"].shape == (25, 25) and np.linalg.norm(ops["a1"] @ ops["a2_dag"] - ops["a2_dag"] @ ops["a1"]) < 1e-14
        passed += 1
        print("[PASS] TODO 2 : build_composite_bipartite_operators")

        H_rwa = build_coupled_ho_hamiltonian(ops, omega1=1.0, omega2=1.0, g_coupling=0.15, include_counter_rotating=False)
        evals_h = np.sort(np.linalg.eigvalsh(H_rwa))
        assert np.isclose(evals_h[1], 0.85) and np.isclose(evals_h[2], 1.15)
        passed += 1
        print("[PASS] TODO 3 : build_coupled_ho_hamiltonian")

        c_ops = build_collapse_operators(ops, kappa1=0.06, kappa2=0.03, n_th1=0.15, n_th2=0.02, gamma_phi1=0.02, gamma_phi2=0.01)
        assert len(c_ops) == 6 and c_ops[0].shape == (25, 25)
        passed += 1
        print("[PASS] TODO 4 : build_collapse_operators")

        psi_1 = np.zeros(5, dtype=np.complex128); psi_1[2] = 1.0
        psi_2 = np.zeros(5, dtype=np.complex128); psi_2[0] = 1.0
        psi_0 = np.kron(psi_1, psi_2)
        rho_0 = np.outer(psi_0, psi_0.conj())

        u_part = compute_unitary_commutator_part(rho_0, H_rwa)
        assert abs(np.trace(u_part)) < 1e-15
        passed += 1
        print("[PASS] TODO 5 : compute_unitary_commutator_part")

        j_part = compute_quantum_jump_part(rho_0, c_ops)
        assert np.isclose(np.trace(j_part).real, 0.245600, atol=1e-6)
        passed += 1
        print(f"[PASS] TODO 6 : compute_quantum_jump_part (Tr = {np.trace(j_part).real:+.6f})")

        a_part = compute_anticommutator_damping_part(rho_0, c_ops)
        assert np.isclose(np.trace(a_part).real, -0.245600, atol=1e-6)
        assert abs(np.trace(u_part + j_part + a_part)) < 1e-15
        passed += 1
        print(f"[PASS] TODO 7 : compute_anticommutator_damping_part (Tr = {np.trace(a_part).real:+.6f})")

        L_super = build_liouvillian_superoperator(H_rwa, c_ops)
        drho_super = (L_super @ rho_0.ravel()).reshape(25, 25)
        assert np.linalg.norm((u_part + j_part + a_part) - drho_super) < 1e-14
        passed += 1
        print("[PASS] TODO 8 : build_liouvillian_superoperator")

        rho_dt = rk4_step_lindblad(rho_0, H_rwa, c_ops, dt=0.05)
        rho_dt_expm = (expm(L_super * 0.05) @ rho_0.ravel()).reshape(25, 25)
        assert np.linalg.norm(rho_dt - rho_dt_expm) < 1e-7
        passed += 1
        print("[PASS] TODO 9 : rk4_step_lindblad")

        rho_1_sub, s_vn_0 = partial_trace_and_entropy(rho_0, 5, 5)
        assert rho_1_sub.shape == (5, 5) and abs(s_vn_0) < 1e-12
        passed += 1
        print("[PASS] TODO 10: partial_trace_and_entropy")

        traj = propagate_lindblad_trajectory(rho_0, H_rwa, c_ops, ops, t_max=60.0, n_steps=500)
        rho_final_expm = (expm(L_super * 60.0) @ rho_0.ravel()).reshape(25, 25)
        assert np.max(traj["trace_error"]) < 1e-13 and np.linalg.norm(traj["final_rho"] - rho_final_expm) < 1e-6
        passed += 1
        print("[PASS] TODO 11: propagate_lindblad_trajectory")

        rho_ss = compute_liouvillian_steady_state(L_super, 25)
        assert np.isclose(np.trace(ops["n1"] @ rho_ss).real, 0.10775, atol=1e-4)
        passed += 1
        print(f"[PASS] TODO 12: compute_liouvillian_steady_state (<n1>_ss = {np.trace(ops['n1'] @ rho_ss).real:.5f})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Lindblad Master Equation checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_lindblad_checkpoints()
print("Checkpoint suite loaded. Implement TODOs 1-12 above and call `run_lindblad_checkpoints()` to test your code!")

Checkpoint suite loaded. Implement TODOs 1-12 above and call `run_lindblad_checkpoints()` to test your code!
